# Policy Sharing in Multi-Agent Reinforcement Learning

이 프로젝트에서는 `simple_spread_v3` 환경에서 정책 공유 여부를 비교한다.

- Shared policy: 모든 에이전트가 하나의 actor-critic을 사용
- Independent policy: 에이전트마다 별도의 actor-critic을 사용
- 알고리즘: IPPO
- 결과: 평가 보상 그래프와 여러 개의 평가 영상

기존 30,000 agent steps 실험은 평가값의 변동이 커서, 이 버전에서는 학습량과 rollout 크기를 늘리고 PPO 업데이트를 조금 더 안정적으로 수정하였다.


## 1. 환경 설정

처음 실행할 때 필요한 패키지는 프로젝트 폴더의 `requirements.txt`로 설치한다.

```bash
conda create -n marl-small python=3.11 -y
conda activate marl-small
pip install -r requirements.txt
jupyter notebook
```

In [1]:
import copy
import os
import random
from pathlib import Path

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Video, display
from mpe2 import simple_spread_v3
from torch.distributions import Categorical
from tqdm.auto import tqdm

# 이 환경은 작은 MLP와 단일 환경을 사용하므로 CPU가 보통 더 안정적이다.
DEVICE = torch.device("cpu")
print("device:", DEVICE)


device: cpu


## 2. 실험 설정

`FAST_RUN=True`는 코드 확인용이다. 제출용 결과를 만들 때는 `False`로 두고 실행한다.

기존 실험보다 학습량을 3배 늘렸고, 한 번에 수집하는 rollout과 평가 에피소드 수도 늘렸다.


In [2]:
FAST_RUN = False

NUM_AGENTS = 3
MAX_CYCLES = 25
HIDDEN_SIZE = 128

TOTAL_AGENT_STEPS = 6_000 if FAST_RUN else 90_000
ROLLOUT_AGENT_STEPS = 1_500 if FAST_RUN else 3_000
EVAL_INTERVAL = 3_000 if FAST_RUN else 10_000
EVAL_EPISODES = 5 if FAST_RUN else 15
VIDEO_EPISODES = 2 if FAST_RUN else 5

ACTOR_LR = 3e-4
CRITIC_LR = 5e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_COEF = 0.2
VALUE_CLIP_COEF = 0.2
UPDATE_EPOCHS = 5
MINIBATCH_SIZE = 512
ENTROPY_COEF_START = 0.02
ENTROPY_COEF_END = 0.003
VALUE_COEF = 0.5
MAX_GRAD_NORM = 0.5
TARGET_KL = 0.03
REWARD_SCALE = 0.1
CURRICULUM_RATIO = 0.20

Path("models").mkdir(exist_ok=True)
Path("videos").mkdir(exist_ok=True)

print("steps per mode:", TOTAL_AGENT_STEPS)


steps per mode: 90000


## 3. 환경과 신경망

각 에이전트는 가까운 에이전트 1명과 landmark 2개만 관측한다. 공유 정책이 에이전트를 구분할 수 있도록 관측 뒤에 간단한 one-hot agent ID를 추가하였다.

초기 20% 구간은 curriculum stage 0에서 학습하고, 나머지 80%는 원래 환경인 stage 1에서 학습한다.


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_env(render_mode=None):
    return simple_spread_v3.parallel_env(
        N=NUM_AGENTS,
        local_ratio=0.5,
        max_cycles=MAX_CYCLES,
        continuous_actions=False,
        render_mode=render_mode,
        curriculum=True,
        terminate_on_success=True,
        num_agent_neighbors=1,
        num_landmark_neighbors=2,
    )


def set_stage(env, stage):
    env.unwrapped.set_curriculum_stage(stage)


env_test = make_env()
obs_test, _ = env_test.reset(seed=0)
agent_names = list(env_test.possible_agents)
raw_obs_dim = len(obs_test[agent_names[0]])
action_dim = env_test.action_space(agent_names[0]).n
env_test.close()

agent_to_index = {agent: i for i, agent in enumerate(agent_names)}


def prepare_obs(obs, agent):
    # 동일한 shared network가 에이전트별 역할을 구분할 수 있도록 ID를 추가한다.
    agent_id = np.zeros(NUM_AGENTS, dtype=np.float32)
    agent_id[agent_to_index[agent]] = 1.0
    return np.concatenate([np.asarray(obs, dtype=np.float32), agent_id])


obs_dim = raw_obs_dim + NUM_AGENTS


def orthogonal_init(layer, gain=np.sqrt(2)):
    nn.init.orthogonal_(layer.weight, gain)
    nn.init.constant_(layer.bias, 0.0)
    return layer


class ActorCritic(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.actor = nn.Sequential(
            orthogonal_init(nn.Linear(obs_dim, HIDDEN_SIZE)), nn.Tanh(),
            orthogonal_init(nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)), nn.Tanh(),
            orthogonal_init(nn.Linear(HIDDEN_SIZE, action_dim), gain=0.01),
        )
        self.critic = nn.Sequential(
            orthogonal_init(nn.Linear(obs_dim, HIDDEN_SIZE)), nn.Tanh(),
            orthogonal_init(nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)), nn.Tanh(),
            orthogonal_init(nn.Linear(HIDDEN_SIZE, 1), gain=1.0),
        )

        self.actor_optimizer = torch.optim.Adam(
            self.actor.parameters(), lr=ACTOR_LR, eps=1e-5
        )
        self.critic_optimizer = torch.optim.Adam(
            self.critic.parameters(), lr=CRITIC_LR, eps=1e-5
        )

    def act(self, obs, deterministic=False):
        obs = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        logits = self.actor(obs)
        dist = Categorical(logits=logits)
        action = torch.argmax(logits, dim=-1) if deterministic else dist.sample()
        value = self.critic(obs).squeeze(-1)
        return int(action.item()), float(dist.log_prob(action).item()), float(value.item())

    def evaluate_actor(self, obs, actions):
        logits = self.actor(obs)
        dist = Categorical(logits=logits)
        return dist.log_prob(actions), dist.entropy()

    def evaluate_value(self, obs):
        return self.critic(obs).squeeze(-1)


print("agents:", agent_names)
print("raw observation dimension:", raw_obs_dim)
print("observation dimension with agent ID:", obs_dim)
print("number of actions:", action_dim)


agents: ['agent_0', 'agent_1', 'agent_2']
raw observation dimension: 12
observation dimension with agent ID: 15
number of actions: 5


## 4. PPO에 사용할 데이터 처리

In [4]:
def policy_key(mode, agent):
    return "shared" if mode == "shared" else agent


def make_policies(mode):
    if mode == "shared":
        return {"shared": ActorCritic(obs_dim, action_dim).to(DEVICE)}
    return {agent: ActorCritic(obs_dim, action_dim).to(DEVICE) for agent in agent_names}


def finish_trajectory(traj):
    rewards = np.asarray(traj["rewards"], dtype=np.float32)
    values = np.asarray(traj["values"], dtype=np.float32)
    dones = np.asarray(traj["dones"], dtype=np.float32)
    advantages = np.zeros_like(rewards)

    gae = 0.0
    for t in reversed(range(len(rewards))):
        next_value = 0.0 if t == len(rewards) - 1 else values[t + 1]
        non_terminal = 1.0 - dones[t]
        delta = rewards[t] + GAMMA * next_value * non_terminal - values[t]
        gae = delta + GAMMA * GAE_LAMBDA * non_terminal * gae
        advantages[t] = gae

    return {
        "obs": np.asarray(traj["obs"], dtype=np.float32),
        "actions": np.asarray(traj["actions"], dtype=np.int64),
        "log_probs": np.asarray(traj["log_probs"], dtype=np.float32),
        "values": values,
        "advantages": advantages,
        "returns": advantages + values,
    }


def combine_batches(batch_list):
    keys = batch_list[0].keys()
    return {key: np.concatenate([batch[key] for batch in batch_list]) for key in keys}


## 5. Rollout 수집과 PPO 업데이트

PPO 업데이트에는 다음 안정화 방법을 사용한다.

- actor와 critic optimizer 분리
- clipped value loss
- entropy coefficient 감소
- learning rate 감소
- target KL을 이용한 과도한 업데이트 방지


In [5]:
def collect_rollout(mode, policies, start_seed, stage):
    env = make_env()
    set_stage(env, stage)
    collected = {key: [] for key in policies}
    episode_scores = []
    agent_steps = 0
    episode_number = 0

    while agent_steps < ROLLOUT_AGENT_STEPS:
        obs, _ = env.reset(seed=start_seed + episode_number)
        trajectories = {
            agent: {"obs": [], "actions": [], "log_probs": [], "rewards": [], "values": [], "dones": []}
            for agent in env.agents
        }
        raw_returns = {agent: 0.0 for agent in env.agents}

        while env.agents:
            active_agents = list(env.agents)
            actions = {}
            action_info = {}

            for agent in active_agents:
                key = policy_key(mode, agent)
                network_obs = prepare_obs(obs[agent], agent)
                action_info[agent] = policies[key].act(network_obs)
                actions[agent] = action_info[agent][0]

            next_obs, rewards, terminations, truncations, _ = env.step(actions)

            for agent in active_agents:
                action, log_prob, value = action_info[agent]
                done = terminations.get(agent, False) or truncations.get(agent, False)
                trajectories[agent]["obs"].append(prepare_obs(obs[agent], agent))
                trajectories[agent]["actions"].append(action)
                trajectories[agent]["log_probs"].append(log_prob)
                trajectories[agent]["rewards"].append(float(rewards[agent]) * REWARD_SCALE)
                trajectories[agent]["values"].append(value)
                trajectories[agent]["dones"].append(done)
                raw_returns[agent] += float(rewards[agent])

            obs = next_obs
            agent_steps += len(active_agents)

        for agent, traj in trajectories.items():
            collected[policy_key(mode, agent)].append(finish_trajectory(traj))

        episode_scores.append(np.mean(list(raw_returns.values())))
        episode_number += 1

    env.close()
    combined = {key: combine_batches(value) for key, value in collected.items()}
    return combined, agent_steps, float(np.mean(episode_scores))


def ppo_update(policy, batch, training_progress):
    obs = torch.as_tensor(batch["obs"], dtype=torch.float32, device=DEVICE)
    actions = torch.as_tensor(batch["actions"], dtype=torch.long, device=DEVICE)
    old_log_probs = torch.as_tensor(batch["log_probs"], dtype=torch.float32, device=DEVICE)
    old_values = torch.as_tensor(batch["values"], dtype=torch.float32, device=DEVICE)
    returns = torch.as_tensor(batch["returns"], dtype=torch.float32, device=DEVICE)
    advantages = torch.as_tensor(batch["advantages"], dtype=torch.float32, device=DEVICE)

    advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)

    # 학습 후반으로 갈수록 탐색과 learning rate를 조금씩 줄인다.
    remaining = max(0.10, 1.0 - training_progress)
    actor_lr = ACTOR_LR * remaining
    critic_lr = CRITIC_LR * remaining
    for group in policy.actor_optimizer.param_groups:
        group["lr"] = actor_lr
    for group in policy.critic_optimizer.param_groups:
        group["lr"] = critic_lr

    entropy_coef = (
        ENTROPY_COEF_START
        + (ENTROPY_COEF_END - ENTROPY_COEF_START) * training_progress
    )

    indices = np.arange(len(obs))
    actor_losses = []
    critic_losses = []
    entropy_values = []
    kl_values = []

    stop_early = False
    for _ in range(UPDATE_EPOCHS):
        np.random.shuffle(indices)
        epoch_kls = []

        for start in range(0, len(indices), MINIBATCH_SIZE):
            idx = torch.as_tensor(indices[start:start + MINIBATCH_SIZE], device=DEVICE)

            new_log_prob, entropy = policy.evaluate_actor(obs[idx], actions[idx])
            log_ratio = new_log_prob - old_log_probs[idx]
            ratio = torch.exp(log_ratio)

            loss1 = -advantages[idx] * ratio
            loss2 = -advantages[idx] * torch.clamp(
                ratio, 1 - CLIP_COEF, 1 + CLIP_COEF
            )
            actor_loss = torch.maximum(loss1, loss2).mean()
            actor_objective = actor_loss - entropy_coef * entropy.mean()

            policy.actor_optimizer.zero_grad()
            actor_objective.backward()
            nn.utils.clip_grad_norm_(policy.actor.parameters(), MAX_GRAD_NORM)
            policy.actor_optimizer.step()

            value = policy.evaluate_value(obs[idx])
            value_clipped = old_values[idx] + torch.clamp(
                value - old_values[idx],
                -VALUE_CLIP_COEF,
                VALUE_CLIP_COEF,
            )
            value_loss = (value - returns[idx]).pow(2)
            value_loss_clipped = (value_clipped - returns[idx]).pow(2)
            critic_loss = 0.5 * torch.maximum(
                value_loss, value_loss_clipped
            ).mean()

            policy.critic_optimizer.zero_grad()
            (VALUE_COEF * critic_loss).backward()
            nn.utils.clip_grad_norm_(policy.critic.parameters(), MAX_GRAD_NORM)
            policy.critic_optimizer.step()

            with torch.no_grad():
                approx_kl = ((ratio - 1.0) - log_ratio).mean()

            actor_losses.append(float(actor_loss.item()))
            critic_losses.append(float(critic_loss.item()))
            entropy_values.append(float(entropy.mean().item()))
            epoch_kls.append(float(approx_kl.item()))

        mean_epoch_kl = float(np.mean(epoch_kls))
        kl_values.append(mean_epoch_kl)
        if mean_epoch_kl > TARGET_KL:
            stop_early = True
            break

    return {
        "actor_loss": float(np.mean(actor_losses)),
        "critic_loss": float(np.mean(critic_losses)),
        "entropy": float(np.mean(entropy_values)),
        "approx_kl": float(np.mean(kl_values)),
        "early_stop": stop_early,
    }


## 6. 평가와 학습 함수

평가는 매번 같은 초기 seed 집합에서 여러 에피소드로 수행한다. 학습 도중 가장 좋은 평가 결과를 보인 모델을 저장하고, 학습이 끝나면 마지막 모델이 아니라 **best model**을 다시 불러온다.


In [6]:
@torch.no_grad()
def evaluate(mode, policies, episodes=5, seed=10000):
    env = make_env()
    set_stage(env, 1)
    scores = []

    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        returns = {agent: 0.0 for agent in env.agents}

        while env.agents:
            actions = {}
            for agent in env.agents:
                key = policy_key(mode, agent)
                network_obs = prepare_obs(obs[agent], agent)
                actions[agent] = policies[key].act(
                    network_obs, deterministic=True
                )[0]

            obs, rewards, _, _, _ = env.step(actions)
            for agent, reward in rewards.items():
                returns[agent] += float(reward)

        scores.append(np.mean(list(returns.values())))

    env.close()
    return float(np.mean(scores)), float(np.std(scores))


def evaluate_random(episodes=20, seed=20000):
    env = make_env()
    set_stage(env, 1)
    scores = []

    for ep in range(episodes):
        _, _ = env.reset(seed=seed + ep)
        returns = {agent: 0.0 for agent in env.agents}

        while env.agents:
            actions = {
                agent: env.action_space(agent).sample()
                for agent in env.agents
            }
            _, rewards, _, _, _ = env.step(actions)
            for agent, reward in rewards.items():
                returns[agent] += float(reward)

        scores.append(np.mean(list(returns.values())))

    env.close()
    return float(np.mean(scores))


def save_model(mode, policies, seed):
    data = {key: policy.state_dict() for key, policy in policies.items()}
    path = Path("models") / f"{mode}_seed{seed}.pt"
    torch.save(data, path)
    return path


def train(mode, seed=1):
    set_seed(seed)
    policies = make_policies(mode)
    history = []
    total_steps = 0
    next_eval = EVAL_INTERVAL
    rollout_count = 0
    initial_mean, initial_std = evaluate(
        mode, policies, episodes=EVAL_EPISODES, seed=seed * 1000
    )
    best_score = initial_mean
    best_states = {
        key: copy.deepcopy(policy.state_dict())
        for key, policy in policies.items()
    }

    history.append({
        "mode": mode,
        "agent_steps": 0,
        "train_return": np.nan,
        "eval_return": initial_mean,
        "eval_std": initial_std,
        "actor_loss": np.nan,
        "critic_loss": np.nan,
        "entropy": np.nan,
        "approx_kl": np.nan,
    })

    progress = tqdm(total=TOTAL_AGENT_STEPS, desc=mode, unit="agent-step")

    while total_steps < TOTAL_AGENT_STEPS:
        stage = 0 if total_steps < int(TOTAL_AGENT_STEPS * CURRICULUM_RATIO) else 1
        batches, new_steps, train_score = collect_rollout(
            mode,
            policies,
            start_seed=seed * 100000 + rollout_count * 100,
            stage=stage,
        )

        update_info = []
        training_progress = min(1.0, total_steps / TOTAL_AGENT_STEPS)
        for key, policy in policies.items():
            update_info.append(
                ppo_update(policy, batches[key], training_progress)
            )

        total_steps += new_steps
        rollout_count += 1
        progress.update(min(new_steps, TOTAL_AGENT_STEPS - progress.n))

        if total_steps >= next_eval or total_steps >= TOTAL_AGENT_STEPS:
            eval_mean, eval_std = evaluate(
                mode,
                policies,
                episodes=EVAL_EPISODES,
                seed=seed * 1000,
            )

            row = {
                "mode": mode,
                "agent_steps": total_steps,
                "train_return": train_score,
                "eval_return": eval_mean,
                "eval_std": eval_std,
                "actor_loss": np.mean([x["actor_loss"] for x in update_info]),
                "critic_loss": np.mean([x["critic_loss"] for x in update_info]),
                "entropy": np.mean([x["entropy"] for x in update_info]),
                "approx_kl": np.mean([x["approx_kl"] for x in update_info]),
            }
            history.append(row)

            print(
                f"{mode:11s} steps={total_steps:7d}, "
                f"eval={eval_mean:7.2f} ± {eval_std:5.2f}, "
                f"entropy={row['entropy']:.3f}"
            )

            if eval_mean > best_score:
                best_score = eval_mean
                best_states = {
                    key: copy.deepcopy(policy.state_dict())
                    for key, policy in policies.items()
                }
                save_model(mode, policies, seed)

            while next_eval <= total_steps:
                next_eval += EVAL_INTERVAL

    progress.close()

    # 영상과 최종 평가는 마지막 모델이 아니라 가장 좋았던 모델을 사용한다.
    if best_states is not None:
        for key, policy in policies.items():
            policy.load_state_dict(best_states[key])

    # 복원된 best model을 최종 파일로 저장한다.
    save_model(mode, policies, seed)

    best_mean, best_std = evaluate(
        mode, policies, episodes=EVAL_EPISODES, seed=seed * 1000
    )
    print(f"{mode:11s} best model: {best_mean:.2f} ± {best_std:.2f}")

    return policies, pd.DataFrame(history)


## 7. Shared policy와 Independent policy 학습

소규모 과제 범위를 유지하기 위해 seed는 1회 사용한다. 다만 각 평가 시 20개의 고정된 초기 상태를 사용하므로 기존 8회 평가보다 곡선의 변동이 줄어든다.

`FAST_RUN=False`에서는 두 방식 모두 90,000 agent steps를 학습한다. 시간이 충분하면 `TOTAL_AGENT_STEPS`를 120,000~150,000으로 늘릴 수 있다.


In [ ]:
shared_policies, shared_history = train("shared", seed=1)
independent_policies, independent_history = train("independent", seed=1)

results = pd.concat(
    [shared_history, independent_history],
    ignore_index=True,
)
results.to_csv("training_results.csv", index=False)

random_baseline = evaluate_random(episodes=EVAL_EPISODES, seed=20000)
print("random policy baseline:", round(random_baseline, 2))

results


shared:   0%|          | 0/90000 [00:00<?, ?agent-step/s]

shared      steps=  12000, eval= -23.87 ±  7.65, entropy=1.599
shared      steps=  21000, eval= -19.23 ±  5.09, entropy=1.585
shared      steps=  30000, eval= -20.79 ±  7.61, entropy=1.576
shared      steps=  42000, eval= -23.62 ±  7.07, entropy=1.555
shared      steps=  51000, eval= -25.38 ±  8.12, entropy=1.529
shared      steps=  60000, eval= -23.92 ±  7.25, entropy=1.548
shared      steps=  72000, eval= -21.59 ±  7.97, entropy=1.542
shared      steps=  81000, eval= -21.07 ±  7.02, entropy=1.551
shared      steps=  90000, eval= -21.54 ±  7.04, entropy=1.551
shared      best model: -19.23 ± 5.09


independent:   0%|          | 0/90000 [00:00<?, ?agent-step/s]

independent steps=  12000, eval= -54.08 ± 14.49, entropy=1.607
independent steps=  21000, eval= -35.87 ± 12.85, entropy=1.605
independent steps=  30000, eval= -32.28 ± 13.37, entropy=1.601
independent steps=  42000, eval= -30.49 ± 12.64, entropy=1.598


## 8. 학습 결과

점은 20개 평가 에피소드의 평균이고, 옅은 영역은 평가 표준편차이다. 이 환경에서는 보상이 음수이므로 값이 0에 가까울수록 성능이 좋다.


In [ ]:
plt.figure(figsize=(8, 4.5))

for mode, group in results.groupby("mode"):
    group = group.sort_values("agent_steps")
    x = group["agent_steps"].to_numpy()
    mean = group["eval_return"].to_numpy()
    std = group["eval_std"].fillna(0.0).to_numpy()

    plt.plot(x, mean, marker="o", label=mode)
    plt.fill_between(x, mean - std, mean + std, alpha=0.12)

plt.axhline(
    random_baseline,
    linestyle="--",
    linewidth=1.2,
    label=f"random baseline ({random_baseline:.1f})",
)
plt.xlabel("Agent steps")
plt.ylabel("Mean evaluation return")
plt.title("Shared policy vs. independent policy")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150)
plt.show()

summary_rows = []
for mode, group in results.groupby("mode"):
    best_row = group.loc[group["eval_return"].idxmax()]
    summary_rows.append({
        "mode": mode,
        "best_return": best_row["eval_return"],
        "best_step": int(best_row["agent_steps"]),
        "final_logged_return": group.sort_values("agent_steps").iloc[-1]["eval_return"],
    })

summary = pd.DataFrame(summary_rows)
summary


## 9. 평가 영상 저장

학습이 끝난 뒤 복원된 best model을 사용한다. Shared와 Independent는 동일한 초기 seed에서 평가하므로 영상끼리 직접 비교할 수 있다.


In [ ]:
@torch.no_grad()
def save_evaluation_videos(mode, policies, episodes=5, seed=50000):
    saved_files = []
    video_rows = []
    mode_dir = Path("videos") / mode
    mode_dir.mkdir(parents=True, exist_ok=True)

    for ep in range(episodes):
        env = make_env(render_mode="rgb_array")
        set_stage(env, 1)
        obs, _ = env.reset(seed=seed + ep)
        returns = {agent: 0.0 for agent in env.agents}

        path = mode_dir / f"episode_{ep + 1:02d}.mp4"
        writer = imageio.get_writer(path, fps=10)
        writer.append_data(env.render())

        while env.agents:
            actions = {}
            for agent in env.agents:
                key = policy_key(mode, agent)
                network_obs = prepare_obs(obs[agent], agent)
                actions[agent] = policies[key].act(
                    network_obs, deterministic=True
                )[0]

            obs, rewards, _, _, _ = env.step(actions)
            for agent, reward in rewards.items():
                returns[agent] += float(reward)
            writer.append_data(env.render())

        writer.close()
        env.close()

        mean_return = float(np.mean(list(returns.values())))
        saved_files.append(str(path))
        video_rows.append({
            "mode": mode,
            "episode": ep + 1,
            "seed": seed + ep,
            "mean_return": mean_return,
            "video": str(path),
        })

    return saved_files, pd.DataFrame(video_rows)


# 같은 초기 배치로 두 정책을 비교한다.
shared_videos, shared_video_scores = save_evaluation_videos(
    "shared", shared_policies, episodes=VIDEO_EPISODES, seed=50000
)
independent_videos, independent_video_scores = save_evaluation_videos(
    "independent", independent_policies, episodes=VIDEO_EPISODES, seed=50000
)

video_scores = pd.concat(
    [shared_video_scores, independent_video_scores],
    ignore_index=True,
)
video_scores.to_csv("video_results.csv", index=False)

print("shared videos:", shared_videos)
print("independent videos:", independent_videos)
video_scores


## 10. 노트북에서 영상 확인

아래 셀은 각 방식의 첫 번째 평가 영상을 노트북 안에서 보여준다. 나머지 영상은 `videos/shared`, `videos/independent` 폴더에서 확인한다.

In [ ]:
print("Shared policy")
display(Video(shared_videos[0], embed=False, width=500))

print("Independent policy")
display(Video(independent_videos[0], embed=False, width=500))

## 11. 정리

이 실험에서는 동일한 환경과 PPO 설정을 사용하고 정책 파라미터 공유 여부만 변경하였다.

기존 버전과 비교해 학습량을 늘리고, agent ID 입력, PPO value clipping, learning-rate/entropy 감소, best model 복원을 추가하였다. 최종 비교에는 보상 곡선과 같은 초기 상태에서 만든 평가 영상을 함께 사용한다.

실제 결과를 확인한 뒤 아래 내용을 짧게 정리해서 제출한다.

- 어느 방식의 best evaluation return이 더 높았는지
- random policy보다 얼마나 개선되었는지
- 에이전트들이 landmark를 나누어 맡는 행동이 나타났는지
- 특정 방향으로 몰리거나 같은 행동을 반복하는 문제가 있었는지
